# Práctica 05 · Transcripción con GPU en Colab

Este notebook hace **solo la parte que necesita GPU**: transcribir los audios con varios modelos y medir
tiempo, VRAM y dispositivo. Todo lo demás (extracción, métricas, app) corre en la laptop con la caché que sale de aquí.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4** → Guardar.

Sube tus audios a Google Drive, en una carpeta llamada `Practica_05_audios`, con estos nombres:
`t1_entrevista_01.m4a` … `t1_entrevista_04.m4a`, `t2_llamada_01.m4a` … `t2_llamada_04.m4a`
(la extensión puede ser .m4a, .mp3, .wav u .ogg). Opcional: `ruido_ambiente.m4a`, 1 minuto de ruido de fondo para el nivel L3.

## 1. ¿Hay GPU?
Si esta celda falla, no activaste la GPU (ver arriba).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Bajar el repositorio e instalar

In [ ]:
REPO = "https://github.com/karelysscs/Practica_05.git"
RAMA = "main"   # cámbiala si el código todavía está en otra rama
!rm -rf /content/Practica_05 && git clone -q -b $RAMA $REPO /content/Practica_05
%cd /content/Practica_05
!pip install -q -r requirements-colab.txt

## 3. Traer los audios desde Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p data/audio
!cp /content/drive/MyDrive/Practica_05_audios/* data/audio/
# el ruido de fondo (si lo grabaste) se convierte a wav con el nombre que espera config.yaml
!for f in data/audio/ruido_ambiente.*; do [ -e "$f" ] && [ "$f" != data/audio/ruido_ambiente.wav ] && ffmpeg -y -loglevel error -i "$f" -ac 1 -ar 16000 data/audio/ruido_ambiente.wav; done
!ls -la data/audio

## 4. Tarea 1: todos los experimentos de `config.yaml`

Modelos `small` y `large-v3`, GPU y CPU, `float16` e `int8`, con y sin glosario. Cada experimento
corre en su propio proceso para que la VRAM medida sea solo la de ese modelo. Tarda unos 20-30 minutos
(el experimento en CPU con el modelo grande es el más lento: por eso se hace con un solo clip).

In [ ]:
!python scripts/transcribir.py --tarea1

## 5. Tarea 2: escalera de degradación L0-L3 y su transcripción

In [ ]:
!python scripts/degradar.py
!python scripts/transcribir.py --escalera

## 6. Prueba de que se usó la GPU
El dispositivo lo reporta el modelo ya cargado (no lo que pedimos), y la VRAM se mide con NVML para este proceso.

In [ ]:
import pandas as pd, json, glob
filas = [json.load(open(f)) for f in glob.glob("resultados/transcripciones/*/*.json")]
df = pd.DataFrame(filas)[["experimento", "clip", "dispositivo_pedido", "dispositivo_real", "gpu", "vram_pico_mb", "seg_por_minuto"]]
df.groupby("experimento").agg({"dispositivo_real": "first", "gpu": "first", "vram_pico_mb": "max", "seg_por_minuto": "mean"})

## 7. Métricas (opcional aquí: también se pueden calcular en la laptop)

In [ ]:
!python scripts/referencias.py
!python scripts/evaluar.py

## 8. Descargar la caché

Se guarda `resultados.zip` en tu Drive (carpeta `Practica_05_audios`) y además se intenta descargar.
Si Colab se desconecta antes de que lo bajes, búscalo en Drive. Descomprímelo en la carpeta del proyecto
(reemplaza `resultados/`). Los audios **no** se suben a GitHub.

In [ ]:
!zip -qr resultados.zip resultados
# copia en Drive: si Colab se desconecta, el zip no se pierde
!cp resultados.zip /content/drive/MyDrive/Practica_05_audios/
from google.colab import files
files.download("resultados.zip")